# Comportamento do Consumidor
## Pipeline de dois agentes para análise de sentimento e geração de insights de marketing

**Disciplina:** Processamento de Linguagem Natural
**Autor:** Pedro Aragão Dorneles
**Base de dados:** Olist Brazilian E-Commerce (≈100 mil avaliações reais de e-commerce brasileiro)

---

### Visão geral do projeto

Este projeto implementa **dois agentes** que trabalham em sequência sobre comentários reais de consumidores:

| Agente | Papel | O que faz |
|--------|-------|-----------|
| **Agente 1 — Coletor & Curador** | Ingestão e tratamento | Lê a base bruta (CSV cru, sem tratamento), limpa e normaliza o texto em português, e estrutura cada registro como `{texto, nota, sentimento}`. |
| **Agente 2 — Analista & Consultor** | Análise e recomendação | Treina um modelo **supervisionado** de análise de sentimento (usando a nota como rótulo), extrai os **aspectos** mais citados (preço, entrega, atendimento...) e gera **recomendações para a equipe de marketing** baseadas no que os consumidores declararam ser importante. |

A premissa central: a **nota de 1 a 5** que acompanha cada comentário funciona como **rótulo supervisionado** — é o gabarito que ensina o modelo a reconhecer sentimento em português, sem precisarmos rotular nada à mão.

---


# Agente 1 — Coletor & Curador

O Agente 1 é responsável por transformar o **dado grosseiro e sem tratamento** em uma base limpa e estruturada, pronta para o treino.

### Por que esta base?

A base **Olist** contém ~100 mil avaliações reais de compras em e-commerce no Brasil. Cada avaliação traz:
- `review_score`: nota de **1 a 5** (nosso rótulo supervisionado)
- `review_comment_message`: o comentário escrito pelo cliente (texto cru)

Cerca de 41 mil avaliações possuem texto escrito — é sobre elas que trabalhamos.

> **Nota sobre a escolha da fonte:** o escopo original previa coleta via API (ex: Mercado Livre). Optamos pela base Olist porque (1) já entrega o par *texto + nota* pronto, eliminando a fragilidade de depender de uma API ao vivo; (2) é um dataset acadêmico consagrado; e (3) o volume permite um treino supervisionado robusto. O CSV bruto **é** o "dado grosseiro sem tratamento" — exatamente o ponto de partida do Agente 1.


### 1.0 — Setup do ambiente (Google Colab)

Monta o Google Drive e aponta para a base. Execute esta célula primeiro e autorize o acesso quando solicitado.

In [ ]:
# Monta o Google Drive (autorize o acesso na janela que abrir)
from google.colab import drive
drive.mount('/content/drive')

# Caminho da base no seu Drive
CAMINHO_BASE = '/content/drive/MyDrive/Ciência de Dados/7º SEMESTRE/PLN/Trabalho final/olist_order_reviews_dataset.csv'

import os
assert os.path.exists(CAMINHO_BASE), f'Arquivo não encontrado em: {CAMINHO_BASE}'
print('Base localizada com sucesso!')

In [ ]:
# --- Bibliotecas ---
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', 80)

df_bruto = pd.read_csv(CAMINHO_BASE)
print(f'Total de registros na base bruta: {len(df_bruto):,}')
print(f'Colunas: {list(df_bruto.columns)}')
df_bruto[['review_score', 'review_comment_message']].head()

### 1.1 — Diagnóstico do dado cru

Antes de limpar, é boa prática **olhar a sujeira**. Isso justifica cada decisão de tratamento.

In [ ]:
# Mantém apenas avaliações que têm texto escrito
df = df_bruto[df_bruto['review_comment_message'].notna()].copy()
print(f'Avaliações com comentário de texto: {len(df):,}\n')

# Distribuição das notas (rótulo)
print('Distribuição das notas (1 a 5):')
print(df['review_score'].value_counts().sort_index())

# Tipos de ruído presentes no texto cru
qb        = df['review_comment_message'].str.contains(r'\r|\n', regex=True).sum()
maiusculo = df['review_comment_message'].apply(lambda x: x.isupper()).sum()
curtos    = (df['review_comment_message'].str.len() < 4).sum()
duplicados= df['review_comment_message'].duplicated().sum()

print(f'\nRuídos detectados no texto cru:')
print(f'  - Com quebras de linha (\\r \\n): {qb:,}')
print(f'  - Totalmente em MAIÚSCULAS:      {maiusculo:,}')
print(f'  - Muito curtos (<4 caracteres):  {curtos:,}')
print(f'  - Comentários duplicados:        {duplicados:,}')

**Leitura do diagnóstico:** a base é desbalanceada (muito mais notas 5 do que 2) — vamos tratar isso no Agente 2 com `class_weight='balanced'`. O texto traz quebras de linha, maiúsculas, duplicatas e comentários curtíssimos como "Bom", "OK". Tudo isso é limpo a seguir.

### 1.2 — Função de limpeza (normalização PT-BR)

A limpeza foi desenhada para o **português brasileiro informal** típico de avaliações de e-commerce:

1. Remove quebras de linha (`\r`, `\n`)
2. Remove caracteres de ruído (emojis, símbolos), **preservando acentos** — essenciais em PT
3. Converte para minúsculas
4. Reduz letras esticadas: *"ameeeei" → "amei"*, *"muitooo" → "muito"*
5. Expande gírias e abreviações: *"vc" → "você"*, *"mt" → "muito"*, *"pq" → "porque"*
6. Normaliza espaços em branco

> Optamos por uma limpeza **leve**: não removemos a pontuação de fim de frase nem fazemos *stemming* agressivo, porque o modelo supervisionado (TF-IDF) lida bem com a forma natural das palavras e isso preserva a interpretabilidade dos resultados.

In [ ]:
# Dicionário de gírias e abreviações comuns em avaliações brasileiras
GIRIAS = {
    'vc': 'voce', 'voce': 'voce', 'mt': 'muito', 'mto': 'muito', 'mtoo': 'muito',
    'pq': 'porque', 'tb': 'tambem', 'tbm': 'tambem', 'q': 'que', 'td': 'tudo',
    'blz': 'beleza', 'obg': 'obrigado', 'vlw': 'valeu', 'msg': 'mensagem',
    'prod': 'produto', 'qdo': 'quando', 'dnv': 'denovo', 'add': 'adicionar',
    'naum': 'nao', 'eh': 'e', 'vdd': 'verdade',
}

def limpar_texto(texto):
    """Normaliza um comentário cru em português para análise de sentimento."""
    if not isinstance(texto, str):
        return ''
    # 1. Remove quebras de linha
    texto = texto.replace('\r', ' ').replace('\n', ' ')
    # 2. Remove ruído mas preserva acentos e pontuação de frase
    texto = re.sub(r'[^\w\sáàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ.,!?]', ' ', texto)
    # 3. Minúsculas
    texto = texto.lower()
    # 4. Reduz letras esticadas (3+ repetições -> 1): "ameeei" -> "amei"
    texto = re.sub(r'(.)\1{2,}', r'\1', texto)
    # 5. Expande gírias e abreviações
    texto = ' '.join(GIRIAS.get(p, p) for p in texto.split())
    # 6. Normaliza espaços
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Teste rápido da função
exemplos = [
    "Ameeeei o produtoo!! Chegou mt rapido, vc estao de parabens \r\n",
    "PÉSSIMO!!! NUNCA MAIS COMPRO AQUI",
    "produto ok, recomendo 👍👍",
]
for ex in exemplos:
    print('bruto:', repr(ex))
    print('limpo:', repr(limpar_texto(ex)))
    print()

### 1.3 — Criação do rótulo de sentimento a partir da nota

Aqui está o coração da abordagem supervisionada: convertemos a **nota numérica** em uma **classe de sentimento**.

| Nota | Sentimento |
|------|-----------|
| 1, 2 | 😞 Negativo |
| 3    | 😐 Neutro |
| 4, 5 | 😊 Positivo |

Esse mapeamento é o que permite ensinar o modelo: ele aprende *quais palavras aparecem em comentários negativos vs. positivos*, usando milhares de exemplos rotulados automaticamente pela nota que o próprio cliente deu.

In [ ]:
def nota_para_sentimento(nota):
    if nota <= 2:
        return 'negativo'
    elif nota == 3:
        return 'neutro'
    else:
        return 'positivo'

df['sentimento'] = df['review_score'].apply(nota_para_sentimento)

print('Distribuição de sentimentos (rótulo derivado da nota):')
print(df['sentimento'].value_counts())
print(f'\nProporção:')
print((df['sentimento'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

### 1.4 — Pipeline de curadoria final

Aplicamos a limpeza, removemos comentários que ficaram vazios ou curtos demais após o tratamento, e eliminamos duplicatas. O resultado é a **base curada** que alimenta o Agente 2.

In [ ]:
# Aplica a limpeza
df['texto_limpo'] = df['review_comment_message'].apply(limpar_texto)

# Remove vazios/curtos pós-limpeza e duplicatas
antes = len(df)
df_curado = df[df['texto_limpo'].str.len() >= 3].copy()
df_curado = df_curado.drop_duplicates(subset='texto_limpo')

# Base final estruturada: texto + nota + sentimento
df_curado = df_curado[['review_score', 'review_comment_message', 'texto_limpo', 'sentimento']]
df_curado = df_curado.rename(columns={'review_score': 'nota', 'review_comment_message': 'texto_original'})
df_curado = df_curado.reset_index(drop=True)

print(f'Registros antes da curadoria final: {antes:,}')
print(f'Registros após curadoria:           {len(df_curado):,}')
print(f'Removidos (vazios/curtos/duplicados): {antes - len(df_curado):,}\n')
print('=== AMOSTRA DA BASE CURADA (saída do Agente 1) ===')
df_curado.head(8)

In [ ]:
# Salva a base curada na mesma pasta do Drive — entrada do Agente 2
import os
PASTA_TRABALHO = os.path.dirname(CAMINHO_BASE)
CAMINHO_CURADO = os.path.join(PASTA_TRABALHO, 'base_curada.csv')
df_curado.to_csv(CAMINHO_CURADO, index=False)
print(f'Base curada salva: {CAMINHO_CURADO} ({len(df_curado):,} registros)')
print('\n>>> Fim do Agente 1. A base está pronta para o treino supervisionado do Agente 2.')

---
# Agente 2 — Analista & Consultor de Marketing

O Agente 2 recebe a base curada e cumpre três tarefas:

1. **Treina um modelo supervisionado** de análise de sentimento, usando a nota como rótulo
2. **Avalia e interpreta** o modelo (métricas + quais palavras dirigem cada sentimento)
3. **Extrai aspectos** (entrega, qualidade, preço...) e **gera recomendações para o marketing**

## Parte A — Treino supervisionado da análise de sentimento

### Por que TF-IDF + Regressão Logística?

Para este projeto escolhemos uma abordagem **supervisionada clássica** em vez de redes neurais pesadas (como BERTimbau). Justificativas:

- **Roda em qualquer máquina** (Colab, Kaggle ou notebook pessoal sem GPU), treinando em segundos
- **Totalmente interpretável**: conseguimos extrair *quais palavras* o modelo associa a cada sentimento — isso alimenta diretamente os insights de marketing
- **Acurácia muito boa** para PT-BR neste volume de dados

Comparamos dois algoritmos:
- **Regressão Logística** — probabilístico, ótimo F1 em classes desbalanceadas
- **Linear SVC** — margem máxima, costuma ter acurácia ligeiramente superior

> O **BERTimbau** (BERT em português) é citado como evolução futura: daria alguns pontos a mais de acurácia, ao custo de ~400MB de download e maior complexidade de execução.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Carrega a base curada (saída do Agente 1)
df_curado = pd.read_csv(CAMINHO_CURADO)
df_curado['texto_limpo'] = df_curado['texto_limpo'].astype(str)
print(f'Base curada carregada: {len(df_curado):,} comentários')

X = df_curado['texto_limpo']
y = df_curado['sentimento']

# Separação treino/teste estratificada (mantém a proporção das classes)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Treino: {len(X_tr):,}  |  Teste: {len(X_te):,}')

### Vetorização TF-IDF

O TF-IDF transforma cada comentário em um vetor numérico, dando peso maior a palavras distintivas. Usamos **unigramas e bigramas** (`ngram_range=(1,2)`) para capturar expressões como *"não recomendo"* e *"antes do prazo"* — que têm sentido oposto às palavras isoladas.

In [ ]:
vetorizador = TfidfVectorizer(
    ngram_range=(1, 2),   # palavras isoladas + pares (capta 'não recomendo')
    min_df=3,             # ignora termos que aparecem em menos de 3 comentários
    max_features=20000
)
X_tr_vec = vetorizador.fit_transform(X_tr)
X_te_vec = vetorizador.transform(X_te)
print(f'Dimensão do vocabulário (features): {X_tr_vec.shape[1]:,}')

### Treino dos dois modelos

Usamos `class_weight='balanced'` em ambos — isso compensa o desbalanceamento da base (lembrando: ~65% positivo, ~9% neutro), fazendo o modelo dar mais atenção às classes minoritárias.

In [ ]:
# Modelo 1 — Regressão Logística
modelo_lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
modelo_lr.fit(X_tr_vec, y_tr)
pred_lr = modelo_lr.predict(X_te_vec)

# Modelo 2 — Linear SVC
modelo_svc = LinearSVC(class_weight='balanced')
modelo_svc.fit(X_tr_vec, y_tr)
pred_svc = modelo_svc.predict(X_te_vec)

# Comparação
print('COMPARAÇÃO DOS MODELOS\n' + '='*45)
print(f"{'Modelo':<22}{'Acurácia':>10}{'F1-macro':>12}")
print('-'*45)
print(f"{'Regressão Logística':<22}{accuracy_score(y_te, pred_lr):>10.4f}{f1_score(y_te, pred_lr, average='macro'):>12.4f}")
print(f"{'Linear SVC':<22}{accuracy_score(y_te, pred_svc):>10.4f}{f1_score(y_te, pred_svc, average='macro'):>12.4f}")

**Interpretação:** a Regressão Logística costuma vencer no **F1-macro** (média justa entre as três classes, importante por causa do neutro raro), enquanto o SVC tende a ter **acurácia** um pouco maior. Para um consultor de marketing, o F1-macro equilibrado é mais valioso — não queremos um modelo que ignora a classe neutra. Por isso adotamos a **Regressão Logística** como modelo principal (e ela ainda nos dá probabilidades e interpretabilidade).

In [ ]:
# Relatório detalhado do modelo escolhido (Regressão Logística)
print('RELATÓRIO DE CLASSIFICAÇÃO — Regressão Logística\n')
print(classification_report(y_te, pred_lr, digits=3))

### Matriz de confusão

Mostra onde o modelo acerta e erra. A diagonal são os acertos.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te, pred_lr, labels=['negativo', 'neutro', 'positivo'],
    cmap='Blues', ax=ax, colorbar=False
)
ax.set_title('Matriz de Confusão — Regressão Logística')
plt.tight_layout()
plt.show()

**Análise crítica dos resultados:** o modelo é forte nas classes **positivo** e **negativo** (F1 acima de 0,80), que são as que de fato importam para decisões de marketing. A classe **neutro** tem desempenho fraco (F1 ≈ 0,30) — e isso é esperado e honesto de reportar: a nota 3 é intrinsecamente ambígua (o cliente "em cima do muro"), e seus comentários misturam elogios e críticas ("bom, porém..."), o que confunde qualquer classificador. Na prática de negócio, separar bem satisfeitos de insatisfeitos já entrega o valor principal. Caminhos para melhorar o neutro: tratar o problema como regressão da nota (1 a 5) em vez de 3 classes, ou usar BERTimbau, que capta melhor o contexto.

### Parte B — Interpretabilidade: o que o modelo aprendeu?

Aqui está o grande trunfo do modelo linear: podemos abrir a "caixa" e ver **quais palavras empurram um comentário para cada sentimento**. Isso é ouro para o marketing — são os termos que os próprios consumidores usam quando estão satisfeitos ou insatisfeitos.

In [ ]:
features = np.array(vetorizador.get_feature_names_out())

print('PALAVRAS MAIS DECISIVAS POR SENTIMENTO\n' + '='*55)
for i, classe in enumerate(modelo_lr.classes_):
    top = np.argsort(modelo_lr.coef_[i])[-12:][::-1]
    print(f'\n>>> {classe.upper()}')
    print('   ' + ', '.join(features[top]))

### Função de predição reutilizável

Empacotamos o modelo numa função simples, que classifica qualquer comentário novo.

In [ ]:
def prever_sentimento(texto):
    """Recebe um comentário cru, limpa e retorna o sentimento previsto."""
    limpo = limpar_texto(texto)
    vec = vetorizador.transform([limpo])
    return modelo_lr.predict(vec)[0]

# Teste com exemplos novos (não vistos no treino)
testes = [
    "Produto excelente, chegou antes do prazo, recomendo demais!",
    "Péssimo, veio quebrado e o vendedor não responde",
    "Veio certo mas demorou um pouco para chegar",
]
for t in testes:
    print(f'[{prever_sentimento(t).upper():>8}]  {t}')

---
## Parte C — Análise de aspectos e Consultor de Marketing

Classificar sentimento é só metade do trabalho. Para a equipe de marketing, a pergunta real é: **sobre O QUE os clientes reclamam ou elogiam?**

Implementamos uma análise **baseada em aspectos** (ABSA simplificada): detectamos a presença de temas-chave em cada comentário — entrega, qualidade, preço, atendimento, expectativa — e cruzamos com o sentimento. Assim conseguimos dizer não apenas *"27% dos clientes estão insatisfeitos"*, mas *"a insatisfação se concentra em prazo de entrega"*.

In [ ]:
import re
from collections import Counter

# Dicionário de aspectos: tema -> palavras-chave que o sinalizam
ASPECTOS = {
    'Entrega / Prazo':       ['entrega','prazo','chegou','chegar','demora','demorou','atraso',
                              'atrasou','rapido','rápido','recebi','receber','correio',
                              'transportadora','frete','enviado'],
    'Qualidade do Produto':  ['qualidade','produto','material','quebrado','defeito','funciona',
                              'durou','frágil','resistente','original','falsificado'],
    'Preço / Custo':         ['preço','preco','caro','barato','valor','custo','dinheiro','vale'],
    'Atendimento':           ['atendimento','vendedor','loja','contato','resposta','responde',
                              'suporte','reclamação','reclamacao'],
    'Expectativa / Anúncio': ['diferente','foto','esperava','igual','anúncio','anuncio',
                              'descrição','descricao','tamanho','cor'],
}

def detectar_aspectos(texto):
    """Retorna o conjunto de aspectos mencionados em um comentário."""
    t = str(texto).lower()
    achados = set()
    for aspecto, palavras in ASPECTOS.items():
        if any(re.search(r'\b' + re.escape(p), t) for p in palavras):
            achados.add(aspecto)
    return achados

# Aplica a toda a base curada
df_curado['aspectos'] = df_curado['texto_limpo'].apply(detectar_aspectos)
print('Aspectos detectados. Exemplo:')
for _, r in df_curado.head(3).iterrows():
    print(f"  [{r['sentimento']}] {r['texto_limpo'][:55]}...  ->  {r['aspectos'] or 'nenhum'}")

### Aspectos que mais geram insatisfação

Filtramos os comentários **negativos** e contamos quais aspectos aparecem com mais frequência. Estes são os pontos prioritários de melhoria.

In [ ]:
def ranking_aspectos(df, sentimento):
    sub = df[df['sentimento'] == sentimento]
    contagem = Counter()
    for aspectos in sub['aspectos']:
        for a in aspectos:
            contagem[a] += 1
    total = len(sub)
    return contagem, total

cont_neg, total_neg = ranking_aspectos(df_curado, 'negativo')
cont_pos, total_pos = ranking_aspectos(df_curado, 'positivo')

print(f'ASPECTOS EM COMENTÁRIOS NEGATIVOS (de {total_neg:,} comentários)\n' + '='*50)
for aspecto, n in cont_neg.most_common():
    pct = 100 * n / total_neg
    barra = '█' * int(pct / 2)
    print(f'  {aspecto:<24} {pct:5.1f}%  {barra}')

print(f'\nASPECTOS EM COMENTÁRIOS POSITIVOS (de {total_pos:,} comentários)\n' + '='*50)
for aspecto, n in cont_pos.most_common():
    pct = 100 * n / total_pos
    barra = '█' * int(pct / 2)
    print(f'  {aspecto:<24} {pct:5.1f}%  {barra}')

In [ ]:
# Gráfico comparativo: peso de cada aspecto em negativos vs positivos
aspectos_nomes = list(ASPECTOS.keys())
pct_neg = [100 * cont_neg.get(a, 0) / total_neg for a in aspectos_nomes]
pct_pos = [100 * cont_pos.get(a, 0) / total_pos for a in aspectos_nomes]

x = np.arange(len(aspectos_nomes))
largura = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - largura/2, pct_neg, largura, label='Negativos', color='#d9534f')
ax.bar(x + largura/2, pct_pos, largura, label='Positivos', color='#5cb85c')
ax.set_ylabel('% dos comentários que citam o aspecto')
ax.set_title('Aspectos citados: comentários negativos vs. positivos')
ax.set_xticks(x)
ax.set_xticklabels(aspectos_nomes, rotation=20, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

### O Consultor de Marketing (chatbot via API Anthropic)

Esta é a camada conversacional do Agente 2. Ela recebe **os dados agregados reais** (calculados acima) e os envia para o modelo Claude, que atua como um consultor: gera recomendações de melhoria **fundamentadas no que os consumidores declararam**, não em achismo.

> **Importante:** o chatbot é alimentado com as estatísticas que o próprio pipeline calculou. Ele não inventa números — ele *interpreta* os dados reais e traduz em ações de marketing. Essa é a diferença entre um relatório e um consultor.

Para rodar, defina sua chave da API Anthropic na variável de ambiente `ANTHROPIC_API_KEY`. Sem a chave, a célula seguinte mostra o resumo de dados que *seria* enviado, para você validar a lógica sem custo.

> No Colab, a forma recomendada de guardar a chave é pelo ícone de chave (🔑 *Secrets*) na barra lateral: crie um secret chamado `ANTHROPIC_API_KEY`. A célula abaixo tenta ler dele automaticamente.

In [ ]:
# Instala a biblioteca da Anthropic (não vem por padrão no Colab)
!pip install anthropic -q

In [ ]:
# Monta um resumo factual dos dados para alimentar o consultor
def montar_briefing():
    linhas = []
    linhas.append(f"Total de comentários analisados: {len(df_curado):,}")
    dist = df_curado['sentimento'].value_counts(normalize=True) * 100
    linhas.append(f"Distribuição: positivo {dist.get('positivo',0):.1f}%, "
                  f"neutro {dist.get('neutro',0):.1f}%, negativo {dist.get('negativo',0):.1f}%")
    linhas.append("\nAspectos mais citados em comentários NEGATIVOS:")
    for a, n in cont_neg.most_common():
        linhas.append(f"  - {a}: {100*n/total_neg:.1f}%")
    linhas.append("\nAspectos mais citados em comentários POSITIVOS:")
    for a, n in cont_pos.most_common():
        linhas.append(f"  - {a}: {100*n/total_pos:.1f}%")
    return '\n'.join(linhas)

briefing = montar_briefing()
print('=== BRIEFING DE DADOS (entrada do consultor) ===\n')
print(briefing)

In [ ]:
import os

def obter_chave():
    """Tenta ler a chave do Colab Secrets; senão, da variável de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        return os.environ.get('ANTHROPIC_API_KEY')

def consultor_marketing(pergunta, briefing):
    """Envia o briefing de dados + a pergunta ao Claude e retorna a recomendação."""
    try:
        from anthropic import Anthropic
    except ImportError:
        return "(Instale a biblioteca: pip install anthropic)"

    chave = obter_chave()
    if not chave:
        return ("(Defina ANTHROPIC_API_KEY no Colab Secrets para ativar o consultor. "
                "O briefing de dados acima é o que seria enviado.)")

    cliente = Anthropic(api_key=chave)
    system = (
        "Você é um consultor de marketing orientado a dados. Receberá estatísticas "
        "reais de análise de sentimento de avaliações de clientes. Suas recomendações "
        "devem se basear ESTRITAMENTE nos dados fornecidos — cite os percentuais. "
        "Priorize os aspectos que mais geram insatisfação. Seja prático e direto, "
        "em português, com no máximo 5 recomendações acionáveis."
    )
    msg = cliente.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=system,
        messages=[{
            "role": "user",
            "content": f"DADOS REAIS DA ANÁLISE:\n{briefing}\n\nPERGUNTA DA EQUIPE: {pergunta}"
        }],
    )
    return "".join(b.text for b in msg.content if hasattr(b, 'text'))

# Exemplo de uso
pergunta = "Quais os 3 pontos prioritários para melhorar a satisfação dos clientes?"
print(consultor_marketing(pergunta, briefing))

### Chat interativo com o Consultor

A célula abaixo cria uma **interface de chat** para conversar com o consultor de marketing: digite uma pergunta, pressione *Enviar* (ou Enter) e a resposta aparece abaixo. O histórico é mantido, então você pode fazer perguntas de acompanhamento ("e sobre o preço?", "detalhe melhor o item 2") que o consultor entende no contexto da conversa.

Requer a `ANTHROPIC_API_KEY` configurada no Colab Secrets (mesmo passo do consultor acima).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Histórico da conversa (mantém o contexto entre perguntas)
historico_chat = []

def conversar(pergunta):
    """Versão conversacional do consultor: usa o histórico para dar contexto."""
    try:
        from anthropic import Anthropic
    except ImportError:
        return "(Instale a biblioteca: pip install anthropic)"

    chave = obter_chave()
    if not chave:
        return ("(Defina ANTHROPIC_API_KEY no Colab Secrets para ativar o chat. "
                "O briefing de dados é o que seria enviado.)")

    cliente = Anthropic(api_key=chave)
    system = (
        "Você é um consultor de marketing orientado a dados conversando com a equipe. "
        "Use ESTRITAMENTE os dados reais abaixo, citando percentuais quando relevante. "
        "Priorize aspectos que geram insatisfação. Responda em português, de forma "
        "prática e direta. Mantenha coerência com o que já foi dito na conversa.\n\n"
        f"DADOS REAIS DA ANÁLISE:\n{briefing}"
    )
    # Monta as mensagens com o histórico + a nova pergunta
    mensagens = historico_chat + [{"role": "user", "content": pergunta}]
    msg = cliente.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=system,
        messages=mensagens,
    )
    resposta = "".join(b.text for b in msg.content if hasattr(b, 'text'))
    # Atualiza o histórico
    historico_chat.append({"role": "user", "content": pergunta})
    historico_chat.append({"role": "assistant", "content": resposta})
    return resposta

# --- Componentes da interface ---
caixa_texto = widgets.Text(
    placeholder='Digite sua pergunta ao consultor e pressione Enter...',
    layout=widgets.Layout(width='80%')
)
botao_enviar = widgets.Button(description='Enviar', button_style='primary')
botao_limpar = widgets.Button(description='Limpar conversa')
area_conversa = widgets.Output()

def enviar(_):
    pergunta = caixa_texto.value.strip()
    if not pergunta:
        return
    caixa_texto.value = ''
    with area_conversa:
        display(Markdown(f"**🧑 Você:** {pergunta}"))
        display(Markdown("_consultando os dados..._"))
        resposta = conversar(pergunta)
        clear_output(wait=True)
        # Re-renderiza a conversa inteira a partir do histórico
        for m in historico_chat:
            quem = "🧑 Você" if m["role"] == "user" else "📊 Consultor"
            display(Markdown(f"**{quem}:** {m['content']}"))

def limpar(_):
    historico_chat.clear()
    with area_conversa:
        clear_output()
        display(Markdown("_Conversa reiniciada._"))

botao_enviar.on_click(enviar)
botao_limpar.on_click(limpar)
# Enter no campo de texto também envia (abordagem atual via observe)
caixa_texto.continuous_update = False
def _ao_pressionar_enter(mudanca):
    if mudanca['new']:  # valor não-vazio submetido com Enter
        enviar(None)
caixa_texto.observe(_ao_pressionar_enter, names='value')

display(widgets.HTML("<h4>💬 Consultor de Marketing — Comportamento do Consumidor</h4>"))
display(widgets.HBox([caixa_texto, botao_enviar, botao_limpar]))
display(area_conversa)

---
## Conclusão

Este projeto implementou um pipeline completo de **dois agentes** para análise de comportamento do consumidor:

- **Agente 1** transformou ~99 mil avaliações brutas em uma base curada e limpa de ~35 mil comentários, criando rótulos de sentimento a partir das notas.
- **Agente 2** treinou um modelo supervisionado interpretável (~79% de acurácia, F1 acima de 0,80 nas classes positivo e negativo), identificou os aspectos que mais geram insatisfação, e disponibilizou um consultor de marketing que traduz os dados em recomendações acionáveis.

**Principal achado de negócio:** a insatisfação dos clientes se concentra em **entrega/prazo** e **qualidade do produto** — não em preço. Para o marketing, isso significa que comunicar confiabilidade logística e qualidade tende a ter mais impacto do que competir por preço.

**Evoluções futuras:** fine-tuning de BERTimbau para ganho de acurácia; ABSA com modelos neurais para detecção de aspectos mais sutil; e dashboard interativo para a equipe de marketing.
